In [47]:
import pandas as pd
import numpy as np
import yabplot as yab
from pyvista import Plotter

import pyvista as pv
pv.global_theme.transparent_background = True

def label_for_yab(data: np.ndarray | list) -> dict:
    assert len(data) == 66, "Data must have 66 entries corresponding to atlas labels."

    labels = np.genfromtxt('/home/gchan/kg98_scratch/gchan/Atlases/Tian/tian_atlas/'
                           'yab_labels.csv', dtype=str)

    data = dict(zip(labels[:66], data[:66]))
    return data

def set_yab_scalebar(pl: Plotter, font_size: int = 10, label_format: str = None,
                      show: bool = True, export_path: str = None) -> None:
    sbar = next(iter(pl.scalar_bars.values()))          # get colorbar object
    sbar.GetLabelTextProperty().SetFontSize(font_size)  # set colorbar label font size
    if label_format is not None:
        sbar.SetLabelFormat(label_format)               # control numeric label formatting
    if export_path:
        pl.screenshot(export_path, transparent_background=True)
        show = False
    if show:
        pl.show(jupyter_backend='static')
    pl.close()

tian_dir = "/home/gchan/kg98_scratch/gchan/Atlases/Tian/tian_atlas/surfaces"

In [36]:
gene_df = pd.read_csv("./data/fig3_gene_df_passing_pairs.csv", index_col=0)

display(gene_df.sort_values("correlation", ascending=False).head(20))

,clearance gene,seed,t_max,t_05,correlation,pass_all
risk gene,,,,,,
PPP1R42,SYT11,53.0,820.0,589.0,0.693257,True
RSPO3,TRIM21,31.0,411.0,302.0,0.691297,True
ERICH1,VMP1,16.0,389.0,318.0,0.689110,True
CHMP6,VMP1,33.0,374.0,295.0,0.688942,True
STMN2,VMP1,40.0,427.0,331.0,0.688480,True
TMEM59,VMP1,40.0,423.0,330.0,0.688325,True
ATF7IP2,UBR2,49.0,518.0,363.0,0.686546,True
SNX6,VMP1,40.0,414.0,322.0,0.686084,True
SHANK2,TRIM21,31.0,399.0,240.0,0.685988,True


In [55]:
gene_pairs = {
    ("PPP1R42", "SYT11"),
    ("RSPO3", "TRIM21"),
    ("ERICH1", "VMP1")
}

for risk_gene, clearance_gene in gene_pairs:
    pair_df = gene_df.loc[
        (gene_df.index == risk_gene) & (gene_df["clearance gene"] == clearance_gene),
        ["seed", "correlation"],
    ]
    pair_df = pair_df.dropna(subset=["seed", "correlation"]).copy()
    pair_df = pair_df.sort_values("seed").reset_index(drop=True)

    all_corr = label_for_yab(pair_df["correlation"].to_numpy())

    # Plot all correlations with a grayscale colormap
    pl = yab.plot_cortical(
        data=all_corr, atlas="schaefer_100", display_type='object',
        views=['left_lateral'], figsize=(1200, 600), cmap="Greys",
        vminmax=(0.2, 0.8),
    )
    set_yab_scalebar(
        pl, font_size=16,
        export_path=f"./results/fig3/grays/{risk_gene}_{clearance_gene}_all_cx.png"
    )

    # Plot all correlations with a YlGnBu colormap
    pl = yab.plot_cortical(
        data=all_corr, atlas="schaefer_100", display_type='object',
        views=['left_lateral'], figsize=(1200, 600), cmap="YlGnBu",
        vminmax=(0.2, 0.8),
    )
    set_yab_scalebar(
        pl, font_size=16,
        export_path=f"./results/fig3/all_YlGnBu/{risk_gene}_{clearance_gene}_max_cx.png"
    )

    # Plot only the region with the maximum correlation in red
    max_corr = label_for_yab(
        np.where(
            pair_df["correlation"].eq(pair_df["correlation"].max()).to_numpy(), 1, 0,
        )
    )

    pl = yab.plot_cortical(
        data=max_corr, atlas="schaefer_100", display_type='object',
        views=['left_lateral'], figsize=(1200, 600), cmap="YlOrRd",
        vminmax=(0, 1),
    )
    set_yab_scalebar(
        pl, font_size=16,
        export_path=f"./results/fig3/max_YlOrRd/{risk_gene}_{clearance_gene}_max_cx.png"
    )



In [61]:
max_df = (
    gene_df.reset_index()
    .sort_values("correlation", ascending=False)
    .drop_duplicates(subset=["risk gene", "clearance gene"], keep="first")
    .set_index("risk gene")
)

seed_counts = (
    max_df['seed']
    .value_counts()
    .reindex(np.arange(1, 67), fill_value=0)  # preserve missing seeds as zeros
    .to_numpy()
)

data = label_for_yab(seed_counts)

pl = yab.plot_cortical(
    data=data, atlas="schaefer_100", display_type='object',
    views=['left_lateral', 'left_medial'], figsize=(1200, 600), cmap='YlOrRd',
    vminmax=(0, 3000)
)
set_yab_scalebar(pl, font_size=24,
                   export_path="./results/fig3/seed_counts_cx.png")

pl = yab.plot_subcortical(
    data=data, custom_atlas_path=tian_dir, display_type='object', bmesh_alpha=0.2,
    figsize=(1200, 600), cmap='YlOrRd', views=['left_lateral', 'left_medial'],
    vminmax=(0, 3000)
)
set_yab_scalebar(pl, font_size=24,
                   export_path="./results/fig3/seed_counts_subcx.png")

In [62]:
seed_corr = pd.read_csv('./data/fig3_mean_seed_corr.csv', header=None).to_numpy()
data = label_for_yab(seed_corr)

pl = yab.plot_cortical(
    data=data, atlas="schaefer_100", display_type='object',
    views=['left_lateral', 'left_medial'], figsize=(1200, 600), cmap='YlGnBu',
    vminmax=(np.round(min(data.values()), 2), np.round(max(data.values()), 2))
)
set_yab_scalebar(pl, font_size=24, label_format='%.2f',
                   export_path="./results/fig3/mean_corr_cx.png")

pl = yab.plot_subcortical(
    data=data, custom_atlas_path=tian_dir, display_type='object', bmesh_alpha=0.2,
    figsize=(1200, 600), cmap='YlGnBu', views=['left_lateral', 'left_medial'],
    vminmax=(np.round(min(data.values()), 2), np.round(max(data.values()), 2))
)
set_yab_scalebar(pl, font_size=24, label_format='%.2f',
                   export_path="./results/fig3/mean_corr_subcx.png")